[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C29_Frontier_Interp_Course/02_features/02_features.ipynb)

# 02 · 单义特征与叠加（复现 Toy Models + auto-interp）

目标：用 numpy 从零**复现 Toy Models of Superposition 的相变**（稀疏度驱动模型表示多少特征），度量叠加（W⊤W 干扰、特征维数、容量守恒），分析特征激活统计，做一个 **auto-interp 模拟评分**的玩具闭环。

路线：Elhage toy 超载模型 + 相变 → 特征激活直方图 → 度量叠加(W⊤W / 特征维数 / ΣD≈d) → 单义性度量 → auto-interp 模拟分 → ✏️ 练习 ×4 → 📖 答案 → 🧪 真实特征统计胶囊。

> 心智模型：**叠加 = 用「偶尔干扰」换「多存特征」的买卖；稀疏度是汇率**。容量守恒：ΣD_i ≈ d（瓶颈只有 d 份容量）。

## 1 · 复现 Toy Models of Superposition：稀疏度驱动相变

Elhage 的玩具模型：`m` 个真特征，但只有 `d < m` 维瓶颈。压缩 `h = W x`，解压 `x̂ = ReLU(W⊤h + b)`，**重要性加权**的重建损失。`W ∈ ℝ^{d×m}` 第 `i` 列 = 特征 `i` 占的方向，`‖W_i‖` ≈ 该特征「被表示的强度」。

我们把**稀疏度 S**（特征不激活的概率）从 0 调到 0.99，数被表示的特征数。预期：稠密→只表示最重要的 ~d 个；越稀疏→表示越多（叠加）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

def train_toy(m=20, d=5, S=0.0, importance_decay=0.9, steps=2500, lr=0.05, batch=1024, seed=0):
    '''Elhage toy superposition model。返回 (W[d,m], b[m], importance[m])。
       S = 稀疏度(特征不激活的概率)；importance 几何衰减(前面的特征更重要)。'''
    r = np.random.default_rng(seed)
    imp = importance_decay ** np.arange(m)              # 特征重要性
    W = r.standard_normal((d, m)) * 0.1; b = np.zeros(m)
    mW = np.zeros_like(W); vW = np.zeros_like(W); mb = np.zeros_like(b); vb = np.zeros_like(b)
    for t in range(1, steps + 1):
        active = (r.random((batch, m)) >= S).astype(float)   # 是否激活
        x = active * r.random((batch, m))                    # 激活幅度 ~U(0,1)
        h = x @ W.T                                          # [b,d] 压缩
        pre = h @ W + b                                      # [b,m] 解压前值
        xh = np.maximum(pre, 0.0)                            # ReLU
        resid = (xh - x) * imp                               # 重要性加权残差
        g_pre = (2.0 / batch) * resid * (pre > 0)            # 经 ReLU
        gb = g_pre.sum(0)
        # pre = (x@W.T)@W + b -> dL/dW 有两项（W 既在压缩又在解压）
        gW = h.T @ g_pre + (g_pre @ W.T).T @ x
        for (P, mP, vP, gP) in [(W, mW, vW, gW), (b, mb, vb, gb)]:
            mP[:] = 0.9 * mP + 0.1 * gP; vP[:] = 0.999 * vP + 0.001 * gP ** 2
            P -= lr * (mP / (1 - 0.9 ** t)) / (np.sqrt(vP / (1 - 0.999 ** t)) + 1e-8)
    return W, b, imp

# 健全性：稠密(S=0) 时只表示最重要的 ~d 个特征
W0, b0, imp0 = train_toy(m=20, d=5, S=0.0)
norms0 = np.linalg.norm(W0, axis=0)
print('S=0.00 各特征 ‖W_i‖:', np.round(norms0, 2))
print('被表示特征数(‖W_i‖>0.5) =', int((norms0 > 0.5).sum()), '(应≈d=5)')
assert (norms0 > 0.5).sum() <= 7, '稠密时应只表示 ~d 个最重要特征'
print('✅ 稠密世界：瓶颈只够装 d 个正交特征，其余被丢弃')

In [ ]:
# 相变：稀疏度从 0 -> 0.99，被表示的特征数应单调上升
results = []
for S in [0.0, 0.5, 0.7, 0.9, 0.99]:
    W, b, imp = train_toy(m=20, d=5, S=S, seed=0)
    n_repr = int((np.linalg.norm(W, axis=0) > 0.5).sum())
    results.append((S, n_repr))
    print(f'稀疏度 S={S:.2f} -> 被表示特征数 = {n_repr}/20')
reprs = [n for _, n in results]
assert reprs[0] <= 7 and reprs[-1] >= 16, '越稀疏应表示越多特征（叠加）'
assert reprs == sorted(reprs), '被表示特征数应随稀疏度单调不减'
print('\n✅ 相变复现：稀疏度↑ → 模型敢把 >d 个特征叠加进 d 维空间')

## 2 · 特征激活直方图：单义特征 = 大量 0 + 稀疏正尾

一个健康单义特征的激活分布有鲜明形状：**绝大多数样本激活≈0**（不相关），**少数样本强激活**（概念出现）。

我们生成稀疏特征激活，画直方图（用 `np.histogram` 文本版），验证「零占多数 + 右长尾」。对比一个**多义/稠密**特征（处处激活）的弥散分布。

In [ ]:
def make_feature_activations(n=5000, p_active=0.05, seed=1):
    '''单义特征：以 p_active 激活，激活时幅度~|N(0,1)|*2。其余=0。'''
    r = np.random.default_rng(seed)
    act = np.zeros(n)
    on = r.random(n) < p_active
    act[on] = np.abs(r.standard_normal(on.sum())) * 2
    return act

def hist_text(act, bins=8):
    counts, edges = np.histogram(act, bins=bins)
    for i in range(bins):
        bar = '█' * int(40 * counts[i] / counts.max())
        print(f'[{edges[i]:5.2f},{edges[i+1]:5.2f}) {counts[i]:5d} {bar}')
    return counts

mono = make_feature_activations(p_active=0.05)
print('单义特征激活直方图：')
counts = hist_text(mono)
frac_zero = (mono < 1e-6).mean()
print(f'\n激活≈0 的比例 = {frac_zero:.1%}  (单义特征应大多为0)')
assert frac_zero > 0.9, '单义特征应大多数时候不激活'
assert counts[0] == counts.max(), '最低档(≈0)应是最高频 -> 大量0'
assert counts[-1] < counts[0] / 5, '高激活应是稀疏长尾'
print('✅ 单义特征：大量 0 + 稀疏正尾（dashboard 的标志性形状）')

In [ ]:
# 对比：多义/稠密特征（处处中等激活），分布弥散、零不占多数
dense = np.abs(rng.standard_normal(5000)) + 0.3      # 几乎处处>0
frac_zero_dense = (dense < 1e-6).mean()
print(f'稠密特征 激活≈0 比例 = {frac_zero_dense:.1%}  (远低于单义特征的 >90%)')
assert frac_zero_dense < 0.1, '稠密特征几乎处处激活'
print('✅ 对比成立：单义(稀疏激活) vs 多义/稠密(处处激活) 直方图截然不同')

## 3 · 度量叠加：W⊤W 干扰、特征维数、容量守恒 ΣD≈d

- **Gram 矩阵 `W⊤W`**：对角=被表示强度，非对角=特征间**干扰**。越稀疏(叠加越多)非对角越大。
- **特征维数 `D_i = ‖W_i‖² / Σ_j (Ŵ_i·W_j)²`**：单个特征占多少维。`=1`独占正交维，`=1/2`反足共享。
- **容量守恒**：`Σ_i D_i ≈ d`（瓶颈只有 d 份容量可分）。

In [ ]:
def interference(W):
    '''W⊤W 非对角的平均 |干扰|。'''
    G = W.T @ W
    off = G[~np.eye(G.shape[0], dtype=bool)]
    return np.abs(off).mean(), np.linalg.norm(G)

def feature_dimensionality(W):
    '''Elhage 特征维数 D_i。返回长度 m 的数组。'''
    norms = np.linalg.norm(W, axis=0)
    What = W / (norms + 1e-9)
    D = np.zeros(W.shape[1])
    for i in range(W.shape[1]):
        den = np.sum((What[:, i] @ W) ** 2)          # Σ_j (Ŵ_i·W_j)^2
        D[i] = norms[i] ** 2 / (den + 1e-12)
    return D

print(f"{'稀疏度S':>8s}{'平均|干扰|':>12s}{'‖W⊤W‖_F':>11s}{'ΣD_i':>8s}{'被表示数':>9s}")
for S in [0.0, 0.9, 0.99]:
    W, b, imp = train_toy(m=20, d=5, S=S, seed=0)
    mi, fro = interference(W); D = feature_dimensionality(W)
    print(f'{S:>8.2f}{mi:>12.3f}{fro:>11.2f}{D.sum():>8.2f}{int((np.linalg.norm(W,axis=0)>0.5).sum()):>9d}')
# 容量守恒：ΣD_i ≈ d=5（无论塞多少特征）
W99, _, _ = train_toy(m=20, d=5, S=0.99, seed=0)
assert abs(feature_dimensionality(W99).sum() - 5) < 1.0, 'ΣD_i 应≈瓶颈维 d=5（容量守恒）'
# 干扰随稀疏度上升
i0 = interference(train_toy(m=20,d=5,S=0.0,seed=0)[0])[0]
i99 = interference(W99)[0]
assert i99 > i0, '越稀疏叠加越多 -> 干扰越大'
print('\n✅ ΣD_i≈d 容量守恒成立；干扰随叠加(稀疏度)上升')

## 4 · 单义性度量：学到的特征有多「纯」

拆开后怎么量化一个特征单义？合成数据里我们知道真概念标签，可以算「该特征的高激活样本是否集中于一个概念」。

构造：每个样本属于一个**潜在概念**；一个「好特征」应只对一个概念的样本激活。用 **purity（纯度）** = 高激活样本中占比最大的概念的比例。单义→纯度≈1；多义→纯度≈1/n_concepts。

In [ ]:
def make_labeled(n=4000, n_concepts=8, seed=2):
    r = np.random.default_rng(seed)
    labels = r.integers(0, n_concepts, size=n)
    return labels, n_concepts

def purity(activation, labels, top_frac=0.05):
    '''高激活样本中,占比最大的概念的比例。1=单义,1/C=均匀(多义)。'''
    k = max(1, int(len(activation) * top_frac))
    top = np.argsort(activation)[::-1][:k]
    vals, cnts = np.unique(labels[top], return_counts=True)
    return cnts.max() / k

labels, C = make_labeled()
# 单义特征：只对概念 3 激活
mono_act = np.where(labels == 3, np.abs(rng.standard_normal(len(labels))) + 0.5, 0.0)
# 多义特征：对随机一半概念都激活
poly_act = np.where(np.isin(labels, [1,3,5,7]), np.abs(rng.standard_normal(len(labels))) + 0.5, 0.0)
p_mono = purity(mono_act, labels); p_poly = purity(poly_act, labels)
print(f'单义特征 purity = {p_mono:.2f} (应≈1)')
print(f'多义特征 purity = {p_poly:.2f} (应明显<1, 接近 1/4)')
assert p_mono > 0.95, '单义特征高激活样本应集中于一个概念'
assert p_poly < p_mono - 0.3, '多义特征纯度应明显更低'
print('✅ purity 度量能区分单义 vs 多义特征')

## 5 · auto-interp：模拟评分闭环（解释→预测→对拍）

auto-interp 的精髓不是「写解释」，而是**用解释去预测激活、与真实激活求相关**（simulation score）。

玩具闭环：① 合成一个有**已知触发规则**的特征（在某属性高时激活）；② 从最大激活样本「学」出解释（哪个属性区分高激活样本）；③ 用解释预测激活、与真值求相关。**对的解释忠实度≈1，错的≈0。**

In [ ]:
# 每个'token'有若干属性；真特征在 attr0 高时激活（这是 ground-truth 触发规则）
n_tok = 800
attrs = rng.random((n_tok, 4))
true_act = np.maximum(attrs[:, 0] - 0.5, 0) * 2          # 真规则：attr0>0.5 才激活

def explain_from_top(attrs, act, k=30):
    '''auto-interp 的「解释」步骤：找最能区分高激活样本的属性下标。'''
    order = np.argsort(act)[::-1]
    top, rest = order[:k], order[k:]
    diffs = attrs[top].mean(0) - attrs[rest].mean(0)     # 每个属性在高激活样本里高多少
    return int(np.argmax(diffs))                         # 解释 = 这个属性

def simulate(attrs, explained_attr):
    '''用解释预测激活：照搬规则形式(attr>0.5时激活)。'''
    return np.maximum(attrs[:, explained_attr] - 0.5, 0) * 2

expl = explain_from_top(attrs, true_act)
print(f'auto-interp 学到的解释：特征在 attr{expl} 高时激活 (真相是 attr0)')
assert expl == 0, '解释应正确识别出 attr0 是触发属性'
# 模拟评分：解释预测的激活 vs 真实激活
sim = simulate(attrs, expl)
faith = np.corrcoef(sim, true_act)[0, 1]
print(f'模拟评分(忠实度) = {faith:.3f} (对的解释应≈1)')
assert faith > 0.95, '正确解释应高忠实度'
# 错误解释(用 attr1)的忠实度应很低
faith_wrong = np.corrcoef(simulate(attrs, 1), true_act)[0, 1]
print(f'错误解释(attr1)忠实度 = {faith_wrong:.3f} (应≈0)')
assert faith_wrong < 0.3, '错误解释忠实度应低'
print('✅ 模拟评分区分了好/坏解释 —— 这就是 auto-interp 可证伪的核心')

---
## ✏️ 练习 1：叠加度 vs 稀疏度曲线

实现 `superposition_ratio(W)` = 被表示特征数(`‖W_i‖>0.5`) / 瓶颈维数 `d`。
`>1` 表示叠加（表示了多于 d 个特征）。对一组稀疏度训玩具模型，验证叠加度随稀疏度上升、且稀疏时 `>1`。

In [ ]:
def superposition_ratio(W):
    # TODO: d = W.shape[0]; 被表示数 = (‖各列‖ > 0.5) 的个数; 返回 被表示数 / d
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
ratios = []
for S in [0.0, 0.9, 0.99]:
    W, _, _ = train_toy(m=20, d=5, S=S, seed=0)
    ratios.append(superposition_ratio(W))
print('叠加度(被表示数/d) @ S=[0,0.9,0.99]:', [round(r,2) for r in ratios])
assert ratios[0] <= 1.4, '稠密时叠加度应≈1(无叠加)'
assert ratios[-1] > 1.5, '很稀疏时应叠加(>1.5 倍于 d)'
assert ratios == sorted(ratios), '叠加度应随稀疏度单调不减'
print('✅ 练习 1 通过：稀疏驱动叠加，叠加度可量化')

## ✏️ 练习 2：特征方向恢复（二部匹配）+ 分裂诊断

实现 `recover_and_split(D_learned, F_true)`：`D_learned[d,k]` 列为学到的特征方向，`F_true[m,d]` 行为真特征。
返回 `(平均最大余弦, n_split)`：对每个**真**特征找余弦最近的学到特征(恢复)；`n_split` = 有多少真特征被**≥2 个**学到特征以 `|cos|>0.8` 指向（特征分裂的信号）。

In [ ]:
def recover_and_split(D_learned, F_true, hi=0.8):
    # TODO:
    #   Dn = D_learned 列单位化; Ft = F_true 行单位化
    #   C = |Ft @ Dn|  形状 [m_true, k]
    #   平均最大余弦 = C.max(axis=1).mean()
    #   每个真特征被多少个学到特征以 |cos|>hi 指向 = (C>hi).sum(axis=1)
    #   n_split = 其中 >=2 的真特征个数
    #   返回 (平均最大余弦, n_split)
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
d = 6; m_true = 5
rr = np.random.default_rng(3)
Ft = rr.standard_normal((m_true, d)); Ft /= np.linalg.norm(Ft, axis=1, keepdims=True)
# 学到的字典：完美复制真特征 + 故意让真特征0被复制两次(模拟分裂)
D_learn = np.concatenate([Ft.T, Ft[0:1].T + 0.01*rr.standard_normal((d,1))], axis=1)
D_learn /= np.linalg.norm(D_learn, axis=0, keepdims=True)
mean_cos, n_split = recover_and_split(D_learn, Ft)
print(f'平均最大余弦={mean_cos:.3f}  分裂的真特征数={n_split}')
assert mean_cos > 0.95, '完美字典应高恢复'
assert n_split == 1, '真特征0被复制两次 -> 应检出 1 个分裂'
print('✅ 练习 2 通过：恢复率 + 分裂诊断（合成数据独有的检验）')

## ✏️ 练习 3：激活直方图的「单义形状」判据

实现 `is_monosemantic_shape(act, zero_frac_min=0.8, tail_ratio_max=0.2)`：判断激活分布是否像单义特征——
(a) 激活≈0(`<1e-6`)的比例 ≥ `zero_frac_min`；(b) 最高激活档频数 / 最低档频数 ≤ `tail_ratio_max`（稀疏尾）。两条都满足返回 `True`。用 8 个 bin。

In [ ]:
def is_monosemantic_shape(act, zero_frac_min=0.8, tail_ratio_max=0.2, bins=8):
    # TODO:
    #   zero_frac = (act<1e-6).mean()
    #   counts,_ = np.histogram(act, bins=bins)
    #   tail_ratio = counts[-1]/max(counts[0],1)
    #   返回 (zero_frac>=zero_frac_min) and (tail_ratio<=tail_ratio_max)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
mono = make_feature_activations(p_active=0.04, seed=9)     # 单义
dense = np.abs(rng.standard_normal(5000)) + 0.3            # 稠密/多义
assert is_monosemantic_shape(mono) == True, '稀疏激活应判为单义形状'
assert is_monosemantic_shape(dense) == False, '稠密激活不应判为单义形状'
print('单义特征 ->', is_monosemantic_shape(mono), '| 稠密特征 ->', is_monosemantic_shape(dense))
print('✅ 练习 3 通过：用直方图形状自动筛单义特征')

## ✏️ 练习 4：auto-interp 忠实度评分

实现 `faithfulness(true_act, attrs, explained_attr, thresh=0.5)`：用「解释」(在 `attrs[:,explained_attr]>thresh` 时激活，幅度=属性值减thresh的正部×2)预测激活，返回与 `true_act` 的相关系数。

In [ ]:
def faithfulness(true_act, attrs, explained_attr, thresh=0.5):
    # TODO: sim = max(attrs[:,explained_attr]-thresh, 0)*2; 返回 corrcoef(sim, true_act)[0,1]
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
rr = np.random.default_rng(7)
A = rr.random((600, 3))
ta = np.maximum(A[:, 2] - 0.5, 0) * 2                  # 真规则: attr2
f_right = faithfulness(ta, A, 2)
f_wrong = faithfulness(ta, A, 0)
print(f'对的解释(attr2)忠实度={f_right:.3f}  错的(attr0)={f_wrong:.3f}')
assert f_right > 0.95 and f_wrong < 0.3, '对的解释应≈1, 错的应≈0'
print('✅ 练习 4 通过：模拟评分能给解释打分（auto-interp 核心）')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def superposition_ratio(W):
    d = W.shape[0]
    n_repr = (np.linalg.norm(W, axis=0) > 0.5).sum()
    return n_repr / d

In [ ]:
# 练习 2 参考答案
def recover_and_split(D_learned, F_true, hi=0.8):
    Dn = D_learned / (np.linalg.norm(D_learned, axis=0, keepdims=True) + 1e-9)
    Ft = F_true / (np.linalg.norm(F_true, axis=1, keepdims=True) + 1e-9)
    C = np.abs(Ft @ Dn)                       # [m_true, k]
    mean_cos = C.max(axis=1).mean()
    n_split = int((( C > hi).sum(axis=1) >= 2).sum())
    return mean_cos, n_split

In [ ]:
# 练习 3 参考答案
def is_monosemantic_shape(act, zero_frac_min=0.8, tail_ratio_max=0.2, bins=8):
    zero_frac = (act < 1e-6).mean()
    counts, _ = np.histogram(act, bins=bins)
    tail_ratio = counts[-1] / max(counts[0], 1)
    return bool(zero_frac >= zero_frac_min and tail_ratio <= tail_ratio_max)

In [ ]:
# 练习 4 参考答案
def faithfulness(true_act, attrs, explained_attr, thresh=0.5):
    sim = np.maximum(attrs[:, explained_attr] - thresh, 0) * 2
    return float(np.corrcoef(sim, true_act)[0, 1])

---
## 🧪 真实数据胶囊：真实 SAE 特征的统计画像

下面是几类**真实 SAE 特征**的典型统计（来自 Anthropic / Neuronpedia 公开的 feature dashboard 观察，约数）。用它们体会真实单义特征的「画像」——稀疏激活、可命名、logit 影响连贯。你刚算的 purity / 直方图形状，就是给真特征做这种体检的工具。

In [ ]:
# 真实 SAE 特征的典型统计（示意；来自公开 feature dashboard 的观察量级）
REAL_FEATURES = [
    # (描述, 激活频率, 最大激活样本主题纯度, 是否单义)
    ('Golden Gate Bridge 特征',     0.0003, 0.95, True),
    ('法语文本 特征',               0.012,  0.92, True),
    ('Python 代码缩进 特征',        0.008,  0.90, True),
    ('某多义神经元(对比)',          0.35,   0.28, False),
]
print(f"{'特征':28s}{'激活频率':>10s}{'主题纯度':>9s}{'单义?':>7s}")
for desc, freq, pur, mono in REAL_FEATURES:
    print(f'{desc:28s}{freq:>10.4f}{pur:>9.2f}{str(mono):>7s}')
print('\n观察：单义特征 激活频率低(稀疏!)、主题纯度高(>0.9)；多义神经元 频率高、纯度低。')
print('这正是你在 worked 4/5 量化的两个轴：稀疏激活 + 高纯度 = 单义。')

**🧪 胶囊练习**：实现 `classify_feature(freq, purity, freq_max=0.05, purity_min=0.7)`：根据激活频率与纯度判断特征类型，返回 `'单义'`(频率<freq_max 且 纯度≥purity_min)、`'dead'`(频率<1e-5)、否则 `'多义/可疑'`。

In [ ]:
def classify_feature(freq, purity, freq_max=0.05, purity_min=0.7):
    # TODO: freq<1e-5 -> 'dead'; freq<freq_max 且 purity>=purity_min -> '单义'; 否则 '多义/可疑'
    raise NotImplementedError

In [ ]:
# 自测
assert classify_feature(0.0003, 0.95) == '单义'
assert classify_feature(0.35, 0.28) == '多义/可疑'
assert classify_feature(1e-6, 0.0) == 'dead'
for desc, freq, pur, mono in REAL_FEATURES:
    print(f'{desc:28s} -> {classify_feature(freq, pur)}')
print('✅ 胶囊练习通过：用频率+纯度自动给特征分类（feature dashboard 体检）')

In [ ]:
# 📖 胶囊参考答案
def classify_feature(freq, purity, freq_max=0.05, purity_min=0.7):
    if freq < 1e-5:
        return 'dead'
    if freq < freq_max and purity >= purity_min:
        return '单义'
    return '多义/可疑'

### 小结
- **叠加 = 重要性×稀疏度的买卖**：稠密→只表示最重要的 ~d 个(正交)；稀疏→叠加，表示 >d 个特征。**相变**可复现。
- **度量叠加**：`W⊤W` 非对角=干扰(随叠加上升)；特征维数 `D_i`；**容量守恒 `ΣD_i≈d`**(瓶颈只有 d 份容量)。
- **单义特征签名**：最大激活样本同主题、直方图「大量0+稀疏正尾」、logit 影响连贯；purity 可量化。
- **auto-interp**：解释→**模拟预测激活**→与真值相关(忠实度)；对的≈1 错的≈0。可证伪是关键。
- **拆干净了吗**：feature splitting(粗→细)、absorption(具体特征被吸收)；合成数据用恢复率+分裂诊断直接对拍。
- **深层张力**：特征是「**拆出来的**」(依赖字典)，不一定是「模型在用的」——后续模块用特征时须记住。

下一站：**模块 03 · 电路与 Attribution** —— 把特征/组件连成因果电路，用 attribution patching 规模化定位。